<a href="https://colab.research.google.com/github/Fardous-bp/CNS-doped-Al-interconnect-alloy/blob/main/CNS_Al_6_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install matcalc

!pip install matgl
!pip install seekpath

!pip install crystal-toolkit

In [ ]:
import os
from pymatgen.core import Structure

# include the folder path like ".cif"
my_cif_file = "CuNi2Sn.cif"

if os.path.exists(my_cif_file):
    my_structure = Structure.from_file(my_cif_file)
    print(f"SUCCESS: {my_cif_file} loaded successfully.")
    print(my_structure)
else:
    print(f"ERROR: File '{my_cif_file}' not found. Please check the name in your Colab files tab.")

In [ ]:
 # The value 0.2 adds random noise (in Angstroms) to the atomic sites
cuni2sn_perturbed = my_structure.copy()
cuni2sn_perturbed.perturb(0.2)

# 2. Expand the lattice volume
# Multiplying by 1.2 increases the total cell volume by 20%
cuni2sn_perturbed.scale_lattice(my_structure.volume * 1.2)

# 3. View the results
print("Perturbed CuNi2Sn Structure:")
print(cuni2sn_perturbed)
cuni2sn_perturbed

In [ ]:
import matcalc
from matcalc.utils import UNIVERSAL_CALCULATORS

import pprint
pprint.pprint(list(UNIVERSAL_CALCULATORS))  # calculators that come with bundled with matgl

In [ ]:
from matcalc.utils import MODEL_ALIASES
pprint.pprint(MODEL_ALIASES)  # list of all "aliased" models

In [ ]:
calculator_pbe = matcalc.load_fp("pbe")
# calculator_pbe = matcalc.load_fp("TensorNet-MatPES-PBE-v2025.1-PES")

In [ ]:
# Initialize the Relaxer exactly like the notebook
relax_calc = matcalc.RelaxCalc(
    calculator_pbe,
    optimizer="FIRE",
    relax_atoms=True,
    relax_cell=True,
)

# This should now complete in 1-3 minutes
print("Starting structural optimization...")
data = relax_calc.calc(cuni2sn_perturbed)

# Output results
print(f"Optimization Successful!")
print(f"Final Energy: {data['energy']:.4f} eV")
print(f"Final Optimized Volume: {data['final_structure'].volume:.2f} A^3")

In [ ]:
pprint.pprint(data)

In [ ]:
final_structure_pbe = data["final_structure"]
print(final_structure_pbe)
final_structure_pbe

In [ ]:
from matcalc import ElasticityCalc

# Use the 'final_structure_pbe'
multiplier_GPa = 160.2176621
elastic_calc = ElasticityCalc(calculator_pbe, relax_structure=False)
elastic_results = elastic_calc.calc(final_structure_pbe)

print(f"Bulk Modulus: {elastic_results['bulk_modulus_vrh'] * multiplier_GPa:.2f} GPa")
print(f"Shear Modulus: {elastic_results['shear_modulus_vrh'] * multiplier_GPa:.2f} GPa")

In [ ]:
from matcalc import PhononCalc
import matplotlib.pyplot as plt

# 1. Initialize for the pure structure
phonon_calc = PhononCalc(
    calculator_pbe,
    supercell_matrix=((2, 0, 0), (0, 2, 0), (0, 0, 2)),
    relax_structure=False
)

print("Calculating Phonon Data for Pure CuNi2Sn...")
pure_phonon_data = phonon_calc.calc(final_structure_pbe)

# 2. Access the Phonopy object
ph = pure_phonon_data["phonon"]

# 3. Trigger the DOS calculation (Fixes the AttributeError)
# We use a mesh of 20x20x20 for Q1-level resolution
ph.run_mesh([20, 20, 20])
ph.run_total_dos()

# 4. Extract frequencies and densities
# get_total_dos_dict() returns a dictionary with 'frequency-points' and 'total-dos'
dos_dict = ph.get_total_dos_dict()
freqs = dos_dict['frequency_points']
densities = dos_dict['total_dos']

# 5. Visualization
plt.figure(figsize=(8, 5))
plt.plot(freqs, densities, color='blue', label='Pure CuNi2Sn')
plt.fill_between(freqs, densities, color='blue', alpha=0.2)
plt.ylim(0, 50)
plt.xlim(-1, 12)
plt.title("Phonon Density of States: Pure Baseline", fontsize=14)
plt.xlabel("Frequency (THz)", fontsize=12)
plt.ylabel("Density of States", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


In [ ]:
# First 'Doped' version of structure
sac_al_structure = final_structure_pbe.copy()

# Replace one Sn site (index 12) with Al to simulate doping
sac_al_structure.replace(12, "Al")

# NOW: Run the Relaxation and Elasticity cells again for this NEW structure

In [ ]:
print("Optimizing the Al-doped SAC alloy...")
doped_results = relax_calc.calc(sac_al_structure)
optimized_doped_structure = doped_results["final_structure"]

print(f"Doped Energy: {doped_results['energy']:.4f} eV")

In [ ]:
# Calculate Elastic Moduli for the doped structure
doped_elastic_results = elastic_calc.calc(optimized_doped_structure)

# Unit conversion (eV/A^3 to GPa)
multiplier_GPa = 160.2176621
doped_bulk = doped_elastic_results['bulk_modulus_vrh'] * multiplier_GPa

print(f"Al-Doped Bulk Modulus: {doped_bulk:.2f} GPa")
# Calculate the % change compared to pure structure results

In [ ]:
# 1. Calculate the Shear Modulus for the Doped structure
# We use the same multiplier for GPa conversion
multiplier_GPa = 160.2176621
doped_shear = doped_elastic_results['shear_modulus_vrh'] * multiplier_GPa

print(f"Al-Doped Shear Modulus: {doped_shear:.2f} GPa")
print(f"Pure Shear Modulus: 37.64 GPa")

In [ ]:
from matcalc import PhononCalc
import matplotlib.pyplot as plt

# 1. Initialize for the optimized doped structure
# Ensure we use the exact same supercell_matrix as the pure one (2x2x2)
phonon_calc_doped = PhononCalc(
    calculator_pbe,
    supercell_matrix=((2, 0, 0), (0, 2, 0), (0, 0, 2)),
    relax_structure=False
)

print("Calculating Phonon Data for Al-Doped CuNi2Sn...")
# Use the final relaxed structure from your previous doping run
doped_phonon_data = phonon_calc_doped.calc(optimized_doped_structure)

# 2. Access the Phonopy object
ph_doped = doped_phonon_data["phonon"]

# 3. Trigger the DOS calculation
# High-resolution mesh (20x20x20)
ph_doped.run_mesh([20, 20, 20])
ph_doped.run_total_dos()

# 4. Extract frequencies and densities
dos_dict_doped = ph_doped.get_total_dos_dict()
freqs_doped = dos_dict_doped['frequency_points']
densities_doped = dos_dict_doped['total_dos']

# 5. Visualization for Your Paper
plt.figure(figsize=(8, 5))
plt.plot(freqs_doped, densities_doped, color='red', label='6.25% Al-Doped SAC')
plt.fill_between(freqs_doped, densities_doped, color='red', alpha=0.2)
plt.ylim(0, 50)
plt.xlim(-1, 12)
plt.title("Phonon Density of States: 6.25% Al-Doped CuNi2Sn", fontsize=14)
plt.xlabel("Frequency (THz)", fontsize=12)
plt.ylabel("Density of States", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Access the Pure Phonopy object from previous successful run
ph_pure = pure_phonon_data["phonon"]

# 2. Execute the Thermal Properties calculation
# We use a fine t_step for smooth curves
ph_pure.run_thermal_properties(t_step=5, t_max=600, t_min=0)
tp_pure = ph_pure.get_thermal_properties_dict()

# 3. Extract Values
temps_p = tp_pure['temperatures']

# CRITICAL CORRECTION: Normalize by the 16-atom supercell factor to get per-atom values
num_atoms = 16

free_energy_p = tp_pure['free_energy'] / num_atoms # Normalized Helmholtz Free Energy (kJ/mol·atom)
entropy_p = tp_pure['entropy'] / num_atoms         # Normalized Entropy (J/K/mol·atom)

# 4. Target "Harsh Environment" temperatures
idx_300 = (np.abs(temps_p - 300)).argmin()
idx_473 = (np.abs(temps_p - 473)).argmin()

print(f"--- PURE CuNi2Sn RESULTS (NORMALIZED PER ATOM) ---")
print(f"Free Energy at 300K: {free_energy_p[idx_300]:.4f} kJ/mol·atom")
print(f"Free Energy at 473K (150°C): {free_energy_p[idx_473]:.4f} kJ/mol·atom")
print(f"Entropy at 473K: {entropy_p[idx_473]:.4f} J/K/mol·atom")

In [ ]:
# 1. Access the Doped Phonopy object
ph_d = doped_phonon_data["phonon"]

# 2. Execute the Thermal Properties calculation
ph_d.run_thermal_properties(t_step=5, t_max=600, t_min=0)
tp_doped = ph_d.get_thermal_properties_dict()

# 3. Extract Values
temps_d = tp_doped['temperatures']
num_atoms = 16
free_energy_d = tp_doped['free_energy'] / num_atoms
entropy_d = tp_doped['entropy'] / num_atoms

# 4. Target the same temperatures for a fair comparison
idx_300_d = (np.abs(temps_d - 300)).argmin()
idx_473_d = (np.abs(temps_d - 473)).argmin()

print(f"--- AL-DOPED SAC RESULTS ---")
print(f"Free Energy at 300K: {free_energy_d[idx_300_d]:.4f} kJ/mol·atom")
print(f"Free Energy at 473K (150°C): {free_energy_d[idx_473_d]:.4f} kJ/mol·atom")
print(f"Entropy at 473K: {entropy_d[idx_473_d]:.4f} kJ/mol·atom")

In [ ]:
import matplotlib.pyplot as plt

# Data from your results
temps = tp_pure['temperatures']
fe_pure = tp_pure['free_energy']
fe_doped = tp_doped['free_energy']

# 1. Plotting
plt.figure(figsize=(9, 6))
plt.plot(temps, fe_pure, color='blue', linewidth=2.5, label='Pure CuNi2Sn')
plt.plot(temps, fe_doped, color='red', linewidth=2.5, linestyle='--', label='Al-Doped SAC Alloy')

# 2. Styling
plt.title("Thermodynamic Stability: Pure vs. Al-Doped SAC", fontsize=15, fontweight='bold')
plt.xlabel("Temperature (K)", fontsize=13)
plt.ylabel("Helmholtz Free Energy (kJ/mol)", fontsize=13)
plt.axvline(x=473, color='gray', linestyle=':', alpha=0.7) # Harsh environment marker
plt.text(480, -100, 'Harsh Environment (150°C)', fontsize=10, color='gray')

plt.legend(fontsize=12)
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()